# What is the evidential strength of "this file existed by T"?

A cryptographic anchor date **T** proves a file existed by T, which freezes the generator
hypothesis space to **G≤T**. This notebook builds the two layers `CLAUDE.md` specifies but the
repo has never had in code — a **calibration layer** turning detector scores into likelihood
ratios, and an **aggregator over G≤T** — and then prices the timestamp: for each image it traces
`LR(F,T)` across dates and reports the crossing point **T\*(F)** where the evidence for
fabrication first reaches a threshold θ.

The output is a likelihood ratio, never a posterior. There is no prior on synthetic prevalence
anywhere in it, by construction (§4).

Nothing here re-runs a detector — it reads `scores/*.csv`, which `./run_all.sh` produced. The
machinery lives in **[`lr/`](lr/)**, documented in [`lr/README.md`](lr/README.md); this notebook
drives it and interprets what comes back. It builds on `merge_scores.ipynb` (the panel, the
σ-rescale idiom), `crop_confound.ipynb` (the crop confound, §3 below), and
`temporal_correlation.ipynb` (the panel is not temporally homogeneous, which is why fusion here is
per-family, §5).

**Three results worth knowing before reading the code.**

1. A **zero-pixel baseline** — the crop-area fraction `200²/(W·H)`, which is metadata, not image
   content — reaches Cllr 0.055 on inpainting and 0.105 on graphics, *better than the five-detector
   panel*. §3 measures this before any LR is computed, so every number afterwards is read as "what
   the panel adds over image geometry."
2. **T\* lands ~2.5–3.3 years before the generator's actual release date, ~90% of the time on the
   early side** — the dangerous direction. §10.
3. Two of six families are represented by a single generator each, so under the repo's own
   unit-of-analysis rule they can support **no likelihood ratio other than 1**, whatever the
   detectors do. §6, §8.

In [ ]:
import numpy as np
import pandas as pd

import lr
from lr import plots, report

plots.style()

Z, zcols = lr.Z, lr.zcols          # the five sigma-score column names
THETAS = (2.0, 4.0, 10.0)          # section 6 shows why nothing above ~16 is supportable here
N_BOOT, N_PERM, SEED = 2000, 2000, 0

print("panel:", ", ".join(lr.DETECTORS))
import sklearn, scipy
print(f"numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn {sklearn.__version__} | scipy {scipy.__version__}")

## 1. The table, and the split

Same join as `merge_scores.ipynb` §1, repeated here so this notebook stands alone rather than
depending on whether `scores/master_scores.csv` is current — and it is not: it predates commit
`ec1b4f6` and carries none of the crop geometry §3 needs.

Two things this join adds over `merge_scores.ipynb`:

- **The `split` column**, which lives only in `data/{authentic,fakes}/*_train.parquet`, never in
  `manifest.csv`. `common.assign_splits` stratified it *by generator*, and its docstring says why:
  "a per-family score model cannot be calibrated on a family that landed entirely on one side."
  That is exactly this notebook, so the split is used as the importers intended.
- **`COCO2017_train` and `COCO2017_val` merged into one `COCO2017` source** (n=204, matching the
  README). `crop_confound.ipynb` §4 keyed on `COCO2017_train` alone and silently dropped 8 images.

`lr.load_scored_manifest` asserts the row count, the per-(family, split) composition, that every
image is scored, and that no detector's score distribution has collapsed.

In [ ]:
M = lr.load_scored_manifest()

print(M.pivot_table(index="generator_family", columns="split", values="image_id",
                    aggfunc="size").to_string())
print()
print(M[M.label == 0].pivot_table(index="source", columns="split", values="image_id",
                                  aggfunc="size").to_string())

### Provenance: the sidecars' manifest hash no longer matches, and that is fine

Every `scores/*.meta.json` records the sha256 of the manifest it scored. That hash no longer
matches `manifest.csv`, because commit `ec1b4f6` **appended** six crop-geometry columns to the
file after the scores were computed. Rather than ignore the mismatch or trust the story,
`lr.provenance` reconstructs the seven columns the detectors actually consumed and hashes those —
which is the claim that has to hold for the cached scores to be usable at all. It asserts the
match, so this cell failing would mean the panel needs re-running.

In [ ]:
live, rebuilt, scored, sidecars = lr.provenance()

print(f"manifest.csv as it stands  {live}")
print(f"the sidecars recorded      {scored}")
print(f"its 7 scored columns only  {rebuilt}")
print("\nthe change since scoring was purely additive: the scores are in sync\n")
print(sidecars.to_string(index=False))

**Reading this:** all five detectors scored the same 1224 rows of the same manifest under the same
`crop200_align16` policy, so the panel is internally consistent. One caveat travels with every
number below and is recorded in `scores/aeroblade.meta.json`: AEROBLADE ran with **2 of its 3
autoencoders**, because `stabilityai/stable-diffusion-2-base` is gated and needs HF auth. Its
scores are `max` over the two that ran.

## 2. Stage 0 — σ-rescale, with the reference fit on authentic **train** rows only

`merge_scores.ipynb` §4 and `temporal_correlation.ipynb` §1 both rescale each detector against
*all 612* authentic images. That is fine for the descriptive plots those notebooks draw, but here
it would leak the validation half into the reference used to fit the calibrators. The fix is one
line inside `lr.sigma_rescale` — restrict the reference to `label == 0 & split == "train"` — and it
returns how much the correction moves, so it is on the record rather than silent.

In [ ]:
M, deltas = lr.sigma_rescale(M)
tr, va = M[M.split == "train"], M[M.split == "val"]
yv = va.label.values

print("max |z_train-referenced - z_all-referenced|, in sigma:")
for k, v in deltas.items():
    print(f"  {k:14s} {v:.4f}")
print(f"\ntrain {len(tr)} rows, val {len(va)} rows")

**Reading this:** the shift is at most **0.16 σ**, so nothing in the existing notebooks'
conclusions turns on it. It is corrected anyway because the calibrators fitted below are far more
sensitive to the reference than a histogram is: a 0.16 σ shift in the authentic reference moves the
PAV block boundaries, and those are what the likelihood ratio is made of.

## 3. The floor: what image geometry alone already buys

`crop_confound.ipynb` §1 found that the crop-area fraction `frac = 200²/(W·H)` separates real from
fake at AUROC 0.85 **with zero pixels** — it is metadata about the source image's resolution, and
native output resolution is a generator fingerprint. That notebook asked whether the detectors
*use* the shortcut and concluded they do not, which is a claim about the sign of a within-class
correlation and is correct as far as it goes.

It is not enough to license likelihood ratios. An LR is a ratio of densities under two
populations, and the denominator `P(s | authentic)` is estimated here on a population whose
covariate distribution does not overlap the numerator's. So before any LR is computed, this
section measures the floor: `frac` pushed through the *identical* PAV/ELUB/Cllr machinery of §4.
Every number after this point is read as "what the panel adds over image geometry."

In [ ]:
fam = lr.family_registry(M)
FAMILIES = list(fam.index)                       # ordered by availability date throughout
FAM_DATE, FAM_NGEN = fam.date.to_dict(), fam.ngen.to_dict()

print("families, ordered by availability date = earliest release among their generators:")
print(fam.assign(date=fam.date.dt.date).to_string())

print("\nlog10(frac) within each fake family:")
print(M[M.label == 1].groupby("generator_family").logfrac
      .agg(lo="min", hi="max", var="var").reindex(FAMILIES).to_string())
print("\nlog10(frac) within each authentic source:")
print(M[M.label == 0].groupby("source").logfrac.agg(n="size", lo="min", hi="max").to_string())

A_LO, A_HI = np.percentile(M.loc[M.label == 0, "logfrac"], [1, 99])
inside = (M[M.label == 1].groupby("generator_family").logfrac
          .apply(lambda s: float(((s >= A_LO) & (s <= A_HI)).mean())))
print(f"\nauthentic 1st-99th pct log10(frac) = [{A_LO:.3f}, {A_HI:.3f}]")
print("share of each fake family inside that range (positivity):")
print(inside.reindex(FAMILIES).round(3).to_string())

In [ ]:
plots.covariate_ecdf(M, FAMILIES, A_LO, A_HI)

**Reading this:** the spec asks for `P(s | family, c)` with `c` a covariate. On this sample that
model is **not identified**, and the plot shows why:

- `inpainting`, `graphics` and `other` have **exactly zero** within-family variance in `c` — every
  inpainting and graphics image is at `frac = 1.0` (the source is already 200×200), and all 34
  `other` images sit at a single value. A conditional density on a point mass is not estimable, and
  this is structural rather than a small-sample problem.
- **RAISE occupies `[-2.60, -2.18]` and overlaps no fake family whatsoever** — every fake family
  stops at `-1.66` or above. RAISE is the genuinely-digitized archival material, the population
  carrying the thesis's central open question, and it is precisely the population that drops out of
  any covariate-matched comparison. That is the sharpest limitation in this notebook.

So `c` is handled by *reporting* rather than conditioning: the floor below, plus the
band-restricted and residualization sensitivities in §11.5–11.6.

## 4. Machinery

The statistics live in [`lr/`](lr/) rather than in this notebook — `lr.core` for the calibrators
and fusion, `lr.bounds` for ELUB, `lr.metrics` for Cllr and the curves. What stays here is the
algebra that makes them defensible, because that is the part that gets defended.
[`lr/README.md`](lr/README.md) carries the same three derivations next to the code.

### The 50/50 construction — why the output carries no prevalence assumption

Give every H_p (fake) row weight `0.5/n_p` and every H_d (authentic) row `0.5/n_d`, so each class
carries total weight exactly ½. A probability estimator fitted on that weighted sample estimates
the posterior of a population whose class prior is ½:

$$\hat p(s)=\frac{\tfrac12 f_p(s)}{\tfrac12 f_p(s)+\tfrac12 f_d(s)}=\frac{f_p(s)}{f_p(s)+f_d(s)}
\quad\Longrightarrow\quad \frac{\hat p(s)}{1-\hat p(s)}=\frac{f_p(s)}{f_d(s)}=\mathrm{LR}(s)$$

Prior odds are 1, so posterior odds *are* the likelihood ratio. Only the weight **ratio** matters,
so `class_weight="balanced"` would be the same construction — but sklearn does not rescale its
penalty by the weights, so `lr.fuse` normalises weights to sum to `n` instead of 1. Otherwise a
given λ would mean different things for a family with 137 fakes and one with 8.

This is discriminative calibration: PAV fits the LR function directly, score → probability → LR. It
never estimates `f_p` and `f_d` separately and divides them.

### Why bounding is mandatory, not cosmetic — an exact identity

The defining property of a properly calibrated likelihood ratio is `E[LR | H_d] = 1`. For
50/50-weighted PAV this **provably fails in-sample**, by a precisely knowable amount. A PAV block
`b` holding `n_pb` fakes and `n_db` authentics gets `LR_b = (n_pb/n_p) / (n_db/n_d)`, so

$$E[\mathrm{LR}\mid H_d]=\sum_b \frac{n_{db}}{n_d}\cdot\mathrm{LR}_b
=\sum_{b:\,n_{db}>0}\frac{n_{pb}}{n_p}=1-\hat P(\mathrm{LR}=\infty\mid H_p)$$

and symmetrically `E[1/LR | H_p] = 1 − P̂(LR = 0 | H_d)`. Any fake mass landing in a block with no
authentics escapes into an `LR = ∞` atom, and the expectation falls short by exactly that mass.
`lr.check_pav_identity` asserts both to 1e-9 on the real per-family data in §6. Unbounded PAV
output is therefore **not** a likelihood ratio, which is what ELUB repairs.

### Cllr and its decomposition

`Cllr` is Brümmer & du Preez's log-likelihood-ratio cost, where **1.0 is the neutral system**
(`LR ≡ 1` for everything, which is what a system with no information should report):

$$C_{llr}=\tfrac12\left[\frac1{N_p}\sum_{i\in H_p}\log_2\!\left(1+\frac1{\mathrm{LR}_i}\right)
+\frac1{N_d}\sum_{j\in H_d}\log_2\!\left(1+\mathrm{LR}_j\right)\right]$$

`Cllr_min` is the same quantity after optimal monotone (PAV) recalibration — the discrimination
component — and `Cllr_cal = Cllr − Cllr_min` is the calibration loss. `Cllr_min` depends only on
the *ordering* of the LRs, so `lr.cllr_min` computes it on ranks, which also makes it immune to the
±∞ atoms. It is always finite: a block with `p = 1` contains no authentics, so every infinite term
is multiplied by an empty average.

One honest caveat: `Cllr_min` here comes from PAV on the same data being scored, so it is
optimistically biased and `Cllr_cal` is an **upper bound** on calibration loss.

### ELUB — the empirical lower and upper bound

Bounding cannot be a taste parameter, so it is derived from a criterion every genuine LR must
satisfy. Markov's inequality applied to the LR itself gives Royall's bound:

$$P(\mathrm{LR}\ge\tau\mid H_d)=\int_{\mathrm{LR}\ge\tau} f_d
=\int_{\mathrm{LR}\ge\tau}\frac{f_p}{\mathrm{LR}}\le\frac1\tau\int f_p\le\frac1\tau$$

ELUB is the empirical enforcement of exactly this, expressed through the **normalised Bayes error
rate**. A threshold τ on the LR is equivalent to prior odds `π = 1/(1+τ)`; against the neutral
system whose error is `min(π, 1−π)`,

$$\mathrm{NBE}(\tau)=\frac{\hat P(\mathrm{LR}\le\tau\mid H_p)+\tau\,\hat P(\mathrm{LR}>\tau\mid H_d)}{\min(1,\tau)}$$

A bound pair `(l, u)` is **admissible** iff the LRs clipped to `[l, u]`, and augmented with one
maximally-misleading observation per class (an H_p case at `l`, an H_d case at `u` — the devil's
advocate saying "suppose the next case is the worst this bound permits"), satisfies `NBE(τ) ≤ 1`
for every τ. `lr.elub` returns the widest admissible pair. `(1, 1)` is always admissible, which is
what a system with nothing supportable returns.

The added misleading observation is what makes the bound finite when no misleading evidence has
actually been *observed*, and it is what makes the bound scale with sample size.

Two cross-checks are reported alongside, and the **tightest of the three is adopted**:

- **`lr.count_bound`** `[1/n_p, n_d]` — you cannot resolve a rate below `1/n`.
- **`lr.source_bound`** `[1/N, N]` with `N` = generators in the family. `temporal_correlation.ipynb`
  §5 already settled that the **generator** is the unit of analysis here (272 diffusion images are
  16 observations, not 272), and the same rule applies to a family's score distribution. For a
  family with one generator this is `[1, 1]`: nothing supportable, at any detector quality.

**Attribution caveat [R]:** the NBE criterion and the devil's-advocate augmentation are recalled
from Vergeer et al. (2016), *Sci. Justice* and the `lir` package's `bounding.py`, and were not
re-read against the paper. The mathematics above stands on its own — it is Markov's inequality —
but the *attribution* should be verified before it goes in the thesis.

### Self-tests

Three controls, because the rest of the notebook is only as trustworthy as these: a shuffled-label
system must land on Cllr ≈ 1 with an ELUB span near zero; a perfect separator must reach
`Cllr_min = 0`; and its ELUB must **saturate at the count bound**, since that is the most any
finite sample can support. `lr.selftest.run` asserts all of it and returns what it measured.

In [ ]:
print(lr.selftest.run(seed=SEED))

In [ ]:
print(f"AUROC(frac alone, all 1224 images) = {lr.auroc(M.frac, M.label):.4f}"
      f"    <- crop_confound.ipynb section 1 reported 0.85")
FLOOR = report.geometry_floor(tr, va, FAMILIES)
print("\nthe geometry-only floor: zero pixels, identical machinery")
print(FLOOR.round(4).to_string())

**Reading this floor:** these are the numbers the five-detector panel has to beat, and for three
families it does not (compare §6). Geometry alone reaches **Cllr 0.055 on inpainting** and
**0.105 on graphics** — both far better than the neutral 1.0, and as §6 shows, better than the
panel manages. The mechanism is not subtle: 16 of the 36 generators emit exactly one output
resolution, so `frac` is close to a generator label for part of the sample.

The direction of the damage matters for this thesis specifically. The families `frac` identifies
best — inpainting (2018-06-10), `other` (2017-07-28), gan (2017-03-30) — are the **early** ones, so
a confound that mimics detection pushes `T*` earlier, which §10 confirms is already the dangerous
direction.

## 5. Stage 1 — fusion, and why it has to be per-family

The panel gives five scores per image; the calibrator needs one. The obvious design is a single
**global** fusion (all fakes vs all authentics) followed by per-family calibration of that one
scalar. It is degenerate, and the reason is worth stating as a proof rather than a measurement:

> PAV output is monotone by construction. If every family's model is `LR_g = φ_g(s)` for the
> *same* scalar `s`, with each `φ_g` non-decreasing, then `max_g LR_g` is itself a non-decreasing
> function of `s`. The aggregator adds no information, `T*(F)` is a relabelling of one number, and
> family attribution is impossible.

So each family gets **its own** ridge-penalised logistic fusion over the five σ-scores — six
parameters against 306 authentic rows plus that family's fakes, which is estimable even at
`n_fake = 8` because the negative class is always 306. λ comes from `lr.pick_l2`: an inner 5-fold
CV *within train*, scored by out-of-fold Cllr with LRs clipped to the count bound (data-independent
given n, so the criterion stays finite without fitting ELUB inside every fold).

This also honours `temporal_correlation.ipynb`'s conclusion — the panel is not temporally
homogeneous, so the calibration layer cannot treat "detector" as exchangeable. Per-family fusion
gives each family its own weight vector over the five detectors; global fusion forces one.

Global fusion is kept as the comparator throughout.

In [ ]:
l2s, cv_curves = {}, {}
for g in FAMILIES:
    l2s[g], cv_curves[g] = lr.pick_l2(tr, g, seed=SEED)

print("lambda by inner 5-fold CV on train (out-of-fold Cllr per lambda):")
print(pd.DataFrame(cv_curves).T.assign(chosen=pd.Series(l2s)).round(4).to_string())

MODS = {"per-family": lr.fit_all(tr, FAMILIES, "per_family", l2s),
        "global": lr.fit_all(tr, FAMILIES, "global")}

In [ ]:
from scipy.stats import rankdata, spearmanr

fused_global, _ = lr.fuse(tr.loc[tr.label == 0, Z].values, tr.loc[tr.label == 1, Z].values, 1.0)
s_global_va = fused_global(va[Z].values)

rows = []
for mode, mm in MODS.items():
    L = lr.lr_matrix(mm, va, FAMILIES)
    R = np.corrcoef(np.apply_along_axis(rankdata, 0, L), rowvar=False)
    off = R[~np.eye(len(FAMILIES), dtype=bool)]
    rows.append({"fusion": mode, "AUROC(max_g LR_g)": lr.auroc(L.max(1), yv),
                 "spearman(max, global fused score)": spearmanr(L.max(1), s_global_va).statistic,
                 "pairwise LR_g spearman: min": off.min(), "median": np.median(off)})
print(f"AUROC of the global fused score itself = {lr.auroc(s_global_va, yv):.4f}\n")
print(pd.DataFrame(rows).set_index("fusion").round(4).to_string())

In [ ]:
plots.fusion_heatmaps(MODS, va, FAMILIES)

**Reading this:** under **global** fusion the six family LRs are near-copies of each other —
pairwise Spearman **median 0.95, minimum 0.74**, and `max_g LR_g` correlates with the single global
score at **0.99**. That is the degeneracy the proof predicts, visible as a heatmap that is red
almost everywhere. Under **per-family** fusion the median falls to **0.52** and the minimum to
**−0.30**: the six models genuinely disagree, which is the precondition for the aggregator meaning
anything and for family attribution being possible at all.

One honest note against the change: `AUROC(max_g LR_g)` is slightly *lower* under per-family
fusion (0.832 vs 0.842). AUROC is the wrong statistic here — `CLAUDE.md` says so, and it is driven
by the bottom of the fake-score distribution rather than the upper tail that matters — but the
number should not be hidden. The case for per-family fusion rests on decorrelation and on the Cllr
gains in §6, not on AUROC.

## 6. Stage 2 — the per-family likelihood ratio and its empirical bounds

Each family's calibrator is fitted on train and evaluated on the held-out val half. For `LR_g`, the
competing hypotheses are "this image came from family g" and "this image is authentic" — so H_d is
the authentic class **only**, and other families' fakes are excluded rather than pooled into the
denominator (they are neither hypothesis, and folding them in collapses the bounds).
`lr.family_xy` asserts it, because it is an easy and quiet mistake.

In [ ]:
print("the E[LR|Hd] identity on the real per-family training data (asserted to 1e-9):")
print(report.identity_table(MODS["per-family"], FAMILIES).round(6).to_string())

In [ ]:
FAMTAB = report.family_table(MODS["per-family"], tr, va, FAMILIES, FAM_NGEN, FLOOR)
print("--- per-family fusion (primary) ---")
print(FAMTAB.round(4).to_string())

print("\n--- global fusion (comparator) ---")
print(report.family_table(MODS["global"], tr, va, FAMILIES, FAM_NGEN)
      [["Cllr", "Cllr_min", "Cllr_cal", "span_dex"]].round(4).to_string())

In [ ]:
# resolution of the authentic-train reference is 1/306, so the FPR grid starts there
FPR_GRID = np.linspace(100.0 / 306, 100.0, 400)

plots.calibrator_cost(MODS["per-family"], FAMILIES, FAM_NGEN, FPR_GRID)

print("what each family charges on authentic images to report a given LR:")
print(report.cost_table(MODS["per-family"], FAMILIES, THETAS, FPR_GRID).round(2).to_string())

### What that table says

**The plot and the cost table are the fastest way to see the ceilings.** The table prices every
threshold in the currency that matters — the authentic false-positive rate you must accept to report
it — and two things in it are worth more than the Cllr column.

First, **gan cannot reach LR = 10 at all**: its maximum reachable LR is 9.70, so the largest family
in the sample, the one the panel detects best, tops out just below the threshold. `inpainting` (5.69)
and `autoregressive` (4.03) top out lower still. Under the ELUB bounds only `other`, `diffusion` and
`graphics` ever reach 10, and two of those three have a source-count ceiling below it.

Second, whether the bound binds is family-dependent, so it is worth checking rather than assuming.
For gan (9.70 against an ELUB ceiling of 126) and `autoregressive` (4.03 against 136) the data runs
out long before the bound does. For `inpainting` (5.69 against 5.7) and `graphics` (14.21 against
14.2) the ELUB ceiling is exactly what is stopping them.

**The bounds are the story, and the source-count bound dominates every one of them.** ELUB alone
would let gan reach 126 and diffusion 102; the count bound never binds (306 authentics is plenty);
but the source-count bound caps gan at **13** and diffusion at **16** — the number of generators
each family actually contains. And for `autoregressive` (VQGAN alone) and `graphics`
(FaceSynthetics alone) it is **[1, 1]**: under the repo's own unit-of-analysis rule, those two
families can support **no likelihood ratio other than 1**, whatever the detectors report. That is a
registry-coverage limit, not a detector failure, and no amount of extra images per generator fixes
it.

Two consequences follow immediately:

- **θ cannot be set high.** Nothing above ~16 is supportable by any family here, so the thresholds
  used from §9 onward are θ ∈ {2, 4, 10}. θ = 100 is unreachable and θ = 10 is already at the
  ceiling for four of six families.
- **Per-family fusion earns its place on Cllr, but not everywhere.** It beats global fusion on
  gan (0.502 vs 0.541), `other` (0.251 vs 0.598) and diffusion (0.559 vs 0.695), and loses badly on
  `autoregressive` (1.234 vs 0.776). That is textbook variance: six parameters fitted against 8
  fakes. §11.4's cross-fit is the honest remedy, and it recovers most of the gap.

**Two families are worse than useless.** `inpainting` (Cllr 1.050) and `autoregressive` (1.234)
score **above 1.0**, the neutral system — the calibrated LR is actively misleading for them. And
comparing the last two columns: the panel beats the geometry-only floor for gan (0.502 vs 0.288 —
it does *not*), diffusion (0.559 vs 0.676 — it does), `other` (0.251 vs 0.376 — it does). For
**inpainting the floor is 0.055 and the panel is 1.050**, and for **graphics the floor is 0.105 and
the panel is 1.092**. On three of six families, one metadata field beats the five-detector panel by
an order of magnitude.

## 7. Calibration and misleading evidence, per family

PAV yields only a handful of distinct LR values, so every curve here is a staircase and is drawn as
one — `steps-post` with a marker at each step, never smoothed. The `distinct_LR` column below is
why: 7 distinct values for the small families against 31 for gan and diffusion.

For the same reason DET plots are drawn only for the three families with at least 20 validation
fakes — gan, diffusion and inpainting. At `n_val_fake = 9` the H_p axis moves in 11% jumps and a
curve through five points would be a lie of presentation, so `other`, `autoregressive` and
`graphics` instead get a strip plot of their individual LR values against the authentic spread.

In [ ]:
RME = report.rme_table(MODS["per-family"], va, FAMILIES)
print("rates of misleading evidence (val, ELUB-bounded):")
print(RME.round(4).to_string())

In [ ]:
plots.tippett_grid(MODS["per-family"], va, FAMILIES, Z, FAMTAB)

In [ ]:
plots.det_and_strip(MODS["per-family"], va, FAMILIES, Z, RME, seed=SEED, min_fake=20)

**Reading this:** the rates of misleading evidence are the numbers that would be cross-examined.
Between **10% and 22% of authentic images** are reported at `LR ≥ 1` — nominally supporting
fabrication — for every family, and `autoregressive` reports `LR ≥ 4` on **19.6%** of authentic
images. In the other direction, `RME_p(LR ≤ 1)` reaches **0.64 for inpainting** and **0.67 for
graphics**: two thirds of those families' fakes are reported as no evidence at all, which is the
same fact the Cllr > 1 rows in §6 record.

The strip plot on the right is the honest presentation for the small families: nine dots each,
many of them overlapping, several below `LR = 1` and most inside the blue authentic spread. That
is what "this family is not resolvable on this sample" actually looks like, and it is far harder to
over-read than a five-point DET curve would be. Note also that `inpainting`'s DET curve sits
*above* gan's and diffusion's almost everywhere — worse at every operating point — which is the
same fact its Cllr of 1.050 reports.

## 8. What the family models actually discriminate, and which family the panel is blind to

Two questions. First, does `LR_g` fire on family `g` rather than on "looks synthetic in general"?
The leakage matrix answers that: geometric-mean `LR_g` by *true* family. A usable set of models has
a dominant diagonal. Second, is there a family the panel simply cannot see?

"Blind" is not one condition, so `lr.report.blind_battery` tests it three ways — they fail
differently and all three are reported:

1. **ELUB span** `log10(u/l) = 0` — no LR other than 1 is empirically supportable.
2. **Permutation test on out-of-sample `Cllr_min`** (2000 label shuffles) — *panel*-blind: the
   detectors carry no ordering information about this family.
3. **Source-count bound `N = 1`** — blind by *registry coverage*, independent of detector quality.

A fourth, different failure gets its own column: **confound-blind**, meaning the family has no
common support with the authentic class in the crop-geometry covariate (§3). A family can be
perfectly visible to the panel and still have no population to compare against.

In [ ]:
L_fam = lr.lr_matrix(MODS["per-family"], va, FAMILIES)
LEAK = report.leakage_matrix(L_fam, va, FAMILIES)

print("geometric-mean LR_g by true family (val, per-family fusion):")
print(LEAK.round(2).to_string())
print("\ndoes each family's own model win on its own images, and win above 1?")
print(report.blind_diagnosis(LEAK, va, FAMILIES).round(2).to_string())

In [ ]:
BLIND = report.blind_battery(MODS["per-family"], M, va, FAMILIES, FAM_NGEN,
                             A_LO, A_HI, n_perm=N_PERM, seed=SEED)
print(f"permutation test, {N_PERM} label shuffles:")
print(BLIND.round(4).to_string())

**Reading this.** Two different failures, and they should not be conflated. The *argmax* diagonal
holds for five of six families — only `autoregressive` genuinely leaks, scoring 1.29 on its own
images while **diffusion's model scores 4.39** on them, so VQGAN images are read as diffusion. That
is defensible on the physics (VQGAN's artifacts are closer to the diffusion families the panel was
trained on than to nothing), but it means the model does not identify what it claims to.

The other failure is quieter and worse: `inpainting` scores **0.92 on its own images** and
`graphics` **0.77** — both *below 1*, i.e. their own model treats their own images as mild evidence
*against* the correct hypothesis. Winning the argmax is meaningless when the winning value is below
1. Only gan (16.1), `other` (15.6) and diffusion (7.8) have a diagonal that is both dominant and
above 1.

On blindness: `graphics` is the family that comes closest to failing the permutation test
(`Cllr_min` ≈ 0.92 against a null median of ≈ 1.0, p ≈ 0.05), and `build_manifest.py:110-112`
predicted it in a code comment before any of this was measured:

> `FaceSynthetics` is Microsoft's *rendered* face corpus. Detectors trained on GAN or diffusion
> artifacts have no reason to fire on it, so it must not be pooled with them.

The registry test is blunter and catches two families: `autoregressive` and `graphics` have one
generator each, so `N = 1` and nothing is supportable. And the confound test catches a third pair —
`inpainting` and `graphics` have **zero** common support with the authentic class in `frac`. Three
different failures, three different fixes: more generators per family, a better panel, and a
comparison population that overlaps in geometry.

## 9. Stage 3 — the aggregator over G≤T

`LR(F,T) = max over g in G≤T of LR_g(F)`, where `G≤T` holds the families whose availability date —
the earliest release among their generators — is at or before T. `lr.build_aggregators` computes
four variants on the same date grid, and **monotonicity is asserted on each with the violation
count printed**, because two of them are expected to fail and the failure is the result.

Monotonicity of the primary is a one-line proof: the max of a *fixed* set of reals over a *nested*
increasing family of index sets is non-decreasing. Both hypotheses are load-bearing —
§11.2 breaks the first and the `/|G≤T|` variants break the second.

### One correction to how `max` is described

`CLAUDE.md` calls max-over-generators "monotone, conservative, forensically defensible." The
monotonicity is right and the defensibility is arguable, but "conservative" needs splitting,
because

$$\max_{g\in G_{\le T}}\mathrm{LR}_g=\sup_{\pi\in\Delta(G_{\le T})}\sum_g \pi_g\,\mathrm{LR}_g$$

makes `max` a **profile likelihood ratio** — a supremum over the within-prosecution prior on which
family did it, which is a nuisance parameter. Profile LRs for composite hypotheses are
systematically anti-conservative. So `max` is conservative *against missing a family*, and
anti-conservative *as evidence of fabrication*, hence anti-conservative for `T*`. Those are
opposite senses of the word, and the second is the one that reaches a courtroom.

Multiplicity is bounded by Markov plus a union bound: if each `LR_g` is properly calibrated then
`P(max_{g≤T} LR_g ≥ t | H_d) ≤ |G≤T| / t`. The cell below measures the inflation against that
bound instead of assuming it.

In [ ]:
DATES = np.array(sorted(set(FAM_DATE.values())), dtype="datetime64[ns]")
AGG, AVAIL = lr.build_aggregators(L_fam, DATES, FAM_DATE, FAMILIES)

print("date grid:", [str(d)[:10] for d in DATES])
print("|G<=T|:   ", AVAIL.sum(1).tolist())
print()
n_trans = len(va) * (len(DATES) - 1)
for name, c in AGG.items():
    v = lr.monotone_violations(c)
    print(f"  {name:28s} {v:5d} / {n_trans} monotone violations"
          + ("   MONOTONE" if v == 0 else "   NOT monotone"))

assert lr.monotone_violations(AGG["max"]) == 0, "the primary aggregator must be monotone"
assert lr.monotone_violations(AGG["max / |G|"]) == 0, "a constant divisor preserves monotonicity"
assert lr.monotone_violations(AGG["max / |G<=T|"]) > 0, "expected the T-dependent divisor to fail"

In [ ]:
MULT = report.multiplicity(AGG, AVAIL, DATES, yv, n_boot=N_BOOT, seed=SEED)
print("multiplicity: E[max_{g<=T} LR_g | authentic val] against the Bonferroni bound |G<=T|")
print(MULT.round(3).to_string(index=False))

print("\nCllr of the aggregator at the final date, vs the best single family:")
for name in ("max", "max / |G|"):
    c, cmin, ccal = lr.cllr_decomp(AGG[name][:, -1], yv)
    print(f"  {name:12s} Cllr={c:.4f}  Cllr_min={cmin:.4f}  Cllr_cal={ccal:.4f}")
print(f"  best single family (gan)  Cllr={FAMTAB.loc['gan', 'Cllr']:.4f}")

In [ ]:
plots.aggregator_panel(AGG, MULT, va, DATES, THETAS, yv)

**Reading this.** The two monotone aggregators are `max` and `max / |G|`; the two that divide by
`|G≤T|` fail, and they fail for a reason worth keeping: adding a family whose `LR_g` sits below the
running average *lowers* the aggregate, so an image can become *less* incriminating as more
generators become available. That is incoherent for a quantity meant to answer "what could have
produced this by T," and it is an argument for `max` that has nothing to do with conservatism. A
**constant** divisor `|G|` fixes it, and it is defensible for a second reason:
`max_{g≤T} LR_g / |G| ≤ (1/|G|) Σ_{g∈G} LR_g`, the uniform-prior mixture over the whole registry —
so the Bonferroni-corrected max is dominated by a genuine likelihood ratio.

The multiplicity is real and measured: `E[max | authentic]` climbs from **1.30** at the first date
to **4.21** at the last, against a Bonferroni bound of 6. A properly calibrated LR must have
`E[LR | H_d] = 1`; the aggregator is off by a factor of four purely from taking a maximum over six
correlated comparisons. It sits below the union bound because the six models share the same 306
authentic rows, which correlates their errors.

Aggregation also costs Cllr outright: `max` reaches 0.903 and `max/|G|` 0.750, both **worse than
gan alone at 0.502**. Combining families is not free, and on this sample it is a net loss against
simply using the best-calibrated family model.

### The aggregator uses ELUB bounds — here is what the source-count bound would do

Everything above clips `LR_g` to its **ELUB** bounds, which is what the brief asks for. But §6
argued the *source-count* bound is tighter and more internally consistent, and adopting it changes
the aggregator qualitatively rather than quantitatively: two families would contribute exactly
`LR_g = 1` for every image, so the max can never fall below 1 once they are available. That is a
claim about the construction, so it is measured rather than asserted.

In [ ]:
n_auth_tr = int((tr.label == 0).sum())
L_adopted = np.empty_like(L_fam)
for j, g in enumerate(FAMILIES):
    m = MODS["per-family"][g]
    lo, hi = lr.adopt_bounds((m["lo"], m["hi"]),
                             lr.count_bound(m["n_tr"], n_auth_tr),
                             lr.source_bound(FAM_NGEN[g]))
    L_adopted[:, j] = np.clip(m["model"](va[Z].values), lo, hi)

AGG_ADOPT, _ = lr.build_aggregators(L_adopted, DATES, FAM_DATE, FAMILIES)
print(f"monotone violations (max, source-bounded): {lr.monotone_violations(AGG_ADOPT['max'])}")

print("\nminimum LR(F,T) over ALL val images, by date -- the exculpatory floor:")
print(pd.DataFrame({"T": [str(d)[:10] for d in DATES], "|G<=T|": AVAIL.sum(1),
                    "min LR (ELUB bounds)": AGG["max"].min(0).round(4),
                    "min LR (source bounds)": AGG_ADOPT["max"].min(0).round(4)}
                   ).to_string(index=False))

print("\nE[max | authentic val] at the final date: "
      f"ELUB {AGG['max'][yv == 0, -1].mean():.3f}  "
      f"source-bounded {AGG_ADOPT['max'][yv == 0, -1].mean():.3f}")
rows = []
for th in THETAS:
    te, ta = lr.crossing(AGG["max"], DATES, th), lr.crossing(AGG_ADOPT["max"], DATES, th)
    rows.append({"theta": th,
                 "cross fake (ELUB)": te[yv == 1].notna().mean(),
                 "cross fake (source)": ta[yv == 1].notna().mean(),
                 "cross authentic (ELUB)": te[yv == 0].notna().mean(),
                 "cross authentic (source)": ta[yv == 0].notna().mean()})
print()
print(pd.DataFrame(rows).set_index("theta").round(3).to_string())

**Reading this:** the exculpatory floor is the column to look at. Under ELUB bounds the minimum
`LR(F,T)` over all 612 val images stays well below 1 at every date (0.015 rising to 0.20), so the
system *can* report evidence against fabrication. Under the source-count bound it rises to exactly
**1.0000** from 2020-12-17 — the date `autoregressive` (one generator, bound `[1, 1]`) joins `G≤T`
— and stays there. From that date on `max_{g≤T} LR_g ≥ 1` for **every image, including every
authentic one**, by construction rather than by measurement. That is the cost of combining a max
aggregator with a family that can support nothing: a reader shown `LR = 1.0` on an authentic
photograph will read it as a finding about the photograph, and it is a finding about the registry.

The rest of the comparison cuts the other way, and decisively. Capping gan at 13 instead of 126
brings `E[max | authentic]` from **4.206 down to 1.951** — nearly the 1.0 that a properly
calibrated LR requires, where the ELUB version is off by a factor of four. False crossings on
authentic images fall from 46.1% to **27.5%** at θ=2 and from 6.9% to **2.0%** at θ=10, while
crossings on fakes fall much less (85.6% → 78.1%, 51.0% → 42.2%). So the source bound is both
better calibrated and less misleading, at the price of never being able to exonerate.

That tradeoff is a genuine decision for the thesis and this notebook does not settle it. The brief
asked for ELUB, so ELUB stays primary above; on this evidence the source bound has the stronger
claim, and §11.4's cross-fit is the setting where it should be re-examined.

## 10. T\*(F) — the crossing point

`T*(F)` is the earliest grid date at which `LR(F,T)` reaches θ. `lr.crossing` computes it as
`argmax` over a **boolean** array rather than `idxmax` over floats, so tied dates return the
earliest.

The quantity to watch is the **temporal localization error** `T* − d_g`, where `d_g` is the true
release date of the generator that actually made the image. Negative means the system says
fabrication was plausible *before* the generator existed, which is the direction that convicts on
bad evidence.

In [ ]:
flat = np.isclose(AGG["max"][:, 0], AGG["max"][:, -1])
print(f"curves flat after the first date: {flat.mean():.1%} of val images "
      f"(authentic {flat[yv == 0].mean():.1%}, fake {flat[yv == 1].mean():.1%})")
print(f"mean distinct LR levels per curve: "
      f"{np.mean([len(np.unique(np.round(r, 10))) for r in AGG['max']]):.2f} of {len(DATES)} dates\n")

TRUE_DATES = va.release_date.values[yv == 1]
TSTAR_TAB, TSTAR = report.tstar_table(AGG["max"], DATES, THETAS, TRUE_DATES, yv)
print(TSTAR_TAB.round(3).to_string())

print("\nT*(fake) distribution by threshold:")
print(pd.DataFrame({f"θ={th:g}": TSTAR[th][yv == 1].dt.date.value_counts()
                    for th in THETAS}).fillna(0).astype(int).sort_index().to_string())

In [ ]:
plots.tstar_panel(TSTAR, DATES, THETAS, yv, TRUE_DATES)

### What T\* actually does here

**The error is early, by years, almost always.** At θ=2 the median `T* − d_g` is **−1218 days**
(−3.3 yr) with **90% of fakes on the early side**; at θ=10 it is −856 days with the same 90%. The
system routinely says fabrication was evidentially plausible three years before the generator that
made the image existed. `CLAUDE.md` names early crossings as the dangerous direction, and this is
that failure mode measured rather than predicted.

**The mechanism is a temporal leak in the construction, not noise.** A family joins the registry at
`min(release_date)` over its members, but its score model is fitted on **all** of them. At
T = 2017-03-30 the `gan` model has already been trained on StyleGAN3 images from 2021. §11.2
removes the leak by refitting each family on only the generators available by T, and the median
error improves to −880 days with 86% early — better, and still early.

**The curve is not degenerate, but it is coarse.** Only 17.0% of curves are flat after the first
date, and `T*(fake)` does spread across all six dates at θ=2 (135 / 40 / 38 / 26 / 9 / 14), so the
anchor date is doing real work. But with six families it can only ever resolve to six dates, and
the first family absorbs 44% of the crossings.

**The authentic crossing rate is the number that should stop anyone from using θ=2.** At θ=2,
**46.1% of authentic images** eventually cross — nearly half. At θ=4 it is 27.8%, and only at θ=10
does it fall to 6.9%, where the fake crossing rate is also down to 51.0%. Since §6 showed that
θ=10 is at or above the supportable ceiling for four of six families, there is no threshold on this
sample that is both defensible and useful. That is a finding about the sample, not a tuning problem.

## 11. Sensitivity panel

Every subsection is labelled SENSITIVITY and reports its own delta against the primary. None of
them replaces §9–§10; they bracket it.

### 11.1 SENSITIVITY — generator-level (36-model) aggregator

The granularity comparator. 36 models, one per generator, each fitted on 8–9 training images with
**λ fixed at 1.0** — per-generator CV on 8 images selects noise, so it is not attempted.

Note what happens to the source-count bound at this granularity: **every** generator model has
`N = 1`, so `[1, 1]` applies to all 36. Finer temporal resolution means fewer sources per model and
therefore *less* supportable evidence — a tension the thesis should state outright. The bounds used
below are ELUB only, and that choice is what makes the comparison possible at all.

In [ ]:
gens = lr.generator_registry(M)
GEN_DATE, GKEYS = gens.date.to_dict(), list(gens.index)

GMODS = lr.fit_generator_models(tr, GKEYS, l2=1.0)
GEN_DATES = np.array(sorted(set(GEN_DATE.values())), dtype="datetime64[ns]")
L_gen = lr.lr_matrix(GMODS, va, GKEYS)
AGG_GEN, _ = lr.build_aggregators(L_gen, GEN_DATES, GEN_DATE, GKEYS)

print(f"36 generator models, ELUB span median "
      f"{np.median([np.log10(m['hi'] / m['lo']) for m in GMODS.values()]):.2f} dex")
print(f"monotone violations (max): {lr.monotone_violations(AGG_GEN['max'])}")
print(f"E[max | authentic val] at the final date:  "
      f"family (6 comparisons) {AGG['max'][yv == 0, -1].mean():.3f}   "
      f"generator (36 comparisons) {AGG_GEN['max'][yv == 0, -1].mean():.3f}\n")

rows = []
for th in THETAS:
    tf = lr.crossing(AGG["max"], DATES, th)[yv == 1].reset_index(drop=True)
    tg = lr.crossing(AGG_GEN["max"], GEN_DATES, th)[yv == 1].reset_index(drop=True)
    ef = lr.localization_error(tf, TRUE_DATES)
    eg = lr.localization_error(tg, TRUE_DATES)
    rows.append({"theta": th, "cross fam": tf.notna().mean(), "cross gen": tg.notna().mean(),
                 "median T* fam": str(tf.dropna().median())[:10],
                 "median T* gen": str(tg.dropna().median())[:10],
                 "median err fam (d)": np.median(ef), "median err gen (d)": np.median(eg),
                 "early fam": float(np.mean(ef < 0)), "early gen": float(np.mean(eg < 0))})
print(pd.DataFrame(rows).set_index("theta").round(3).to_string())

**Reading 11.1 — and a correction to what I expected.** The naive intuition, and `CLAUDE.md`'s
warning, is that finer temporal granularity biases crossings *earlier*. On this sample it does the
**opposite**: at θ=2 the median `T*` moves from 2017-03-30 to **2018-06-10**, i.e. *later*, and the
median localization error improves. The reason is that a single generator's model is fitted on 8
images and calibrates weakly, so the earliest generator alone cannot push the LR high — whereas the
pooled 111-image `gan` family can, and does, at the very first date.

That does not vindicate generator granularity. It buys the later median at a steep price:
`E[max | authentic]` rises from **4.21 to 11.09**, because the max now runs over 36 correlated
comparisons instead of 6. A properly calibrated LR needs `E[LR | H_d] = 1`; this is an order of
magnitude out. And the crossing rate on fakes rises to 95.1%, which sounds good until you notice it
rises because *everything* crosses, authentic included.

So the honest label for `|T*_36 − T*_family|` is **the granularity sensitivity of T\***, not a
measurement of the early-crossing bias: both aggregators are biased early, by different mechanisms
(one by multiplicity, one by the pooled-over-future leak of §10), and their difference is a
difference of two biased quantities.

### 11.2 SENSITIVITY — availability-truncated family models

This removes §10's temporal leak. `LR_g^{≤T}` is fitted only on the generators in family `g` whose
release date is ≤ T, so at T = 2017-03-30 the `gan` model knows CycleGAN and nothing else. There
are exactly 36 distinct models to fit — one per generator-addition event — not one per (family,
date) pair.

Truncation breaks the fixed-model premise of §9's monotonicity proof, so the result is wrapped in a
running max and that is stated as a **definition**, not an assumption:

$$\mathrm{LR}(F,T)=\max_{T'\le T}\ \max_{g\in G_{\le T'}}\mathrm{LR}^{\le T'}_g(F)$$

It is coherent because `G≤T' ⊆ G≤T`: evidence supportable at `T'` remains supportable at `T`.

In [ ]:
TRUNC = lr.fit_truncated_models(tr, gens, FAMILIES, l2s)
print(f"{len(TRUNC)} truncated models fitted (one per generator-addition event)")

Xva = va[Z].values
raw = np.full((len(va), len(GEN_DATES)), -np.inf)
for t, T in enumerate(GEN_DATES):
    for g in FAMILIES:
        members = gens[gens.fam == g].sort_values("date")
        j = int((members.date.values <= T).sum())
        if j:
            m = TRUNC[(g, j)]
            raw[:, t] = np.maximum(raw[:, t], np.clip(m["model"](Xva), m["lo"], m["hi"]))
run = np.maximum.accumulate(raw, axis=1)

print(f"before the running max: {lr.monotone_violations(raw)} monotone violations "
      f"(expected -- truncation refits the model)")
print(f"after the running max:  {lr.monotone_violations(run)}")
flat_t = np.isclose(run[:, 0], run[:, -1])
print(f"\nflat-curve share: primary {flat.mean():.1%}  ->  truncated {flat_t.mean():.1%}")
print(f"mean distinct LR levels: primary "
      f"{np.mean([len(np.unique(np.round(r, 10))) for r in AGG['max']]):.2f}  ->  truncated "
      f"{np.mean([len(np.unique(np.round(r, 10))) for r in run]):.2f}\n")

TRUNC_TAB, _ = report.tstar_table(run, GEN_DATES, THETAS, TRUE_DATES, yv)
print(TRUNC_TAB.round(3).to_string())

**Reading 11.2:** this is the version under which the curve carries information. The flat-curve
share collapses from **17.0% to 1.1%** and the mean number of distinct LR levels rises from 2.98 to
4.56, because a family's evidential strength now *grows* as its members are released instead of
arriving fully formed at its first generator's date. The localization error improves at every
threshold — at θ=10, median −550 days with **73% early** against the primary's −856 days and 90%
early.

It is still early. Removing the leak reduces the bias substantially; it does not eliminate it,
because the multiplicity of §9 is untouched.

**And it costs false crossings, which is the part not to skip.** The authentic crossing rate rises
from 46.1% to **51.0%** at θ=2 and from 6.9% to **14.4%** at θ=10 — roughly double at the only
threshold §10 found tolerable. The mechanism is the same one that fixes the localization error:
36 dates instead of 6 means 36 chances for an authentic image to cross, so the multiplicity that
§9 measured over 6 families is now paid over a much finer grid. Truncation fixes the leak and
worsens the multiplicity, and the two effects are separable but both real.

I would still argue this construction, not the primary, is the one the thesis should carry forward —
it is the only one where the LR curve responds to the registry growing, which is the whole premise —
but it needs the multiplicity correction of §9 applied on top, and it is presented as a sensitivity
because it departs from the spec's definition of `G≤T`.

### 11.3 SENSITIVITY — logistic calibration instead of PAV

In [ ]:
CAL_CMP = report.calibrator_comparison(tr, va, FAMILIES, l2s)
print(CAL_CMP.round(4).to_string())
print(f"\nlogistic wins on Cllr for "
      f"{int((CAL_CMP.logistic_Cllr < CAL_CMP.PAV_Cllr).sum())} of {len(FAMILIES)} families")

**Reading 11.3:** `CLAUDE.md` settles on PAV, and the reasoning there is sound — it fits the LR
function directly rather than dividing two separately-estimated densities. But the measurement does
not support PAV unconditionally. Logistic calibration gives **lower Cllr on four of six families**,
and the gap is largest exactly where it should be: `autoregressive` 1.024 vs **1.234** and
`graphics` 0.929 vs **1.092**, the two families with 8 training fakes, where PAV's step function
overfits badly. PAV wins on calibration loss for the large families (`Cllr_cal` 0.039 vs 0.072 on
gan), which is its real advantage.

PAV is kept as primary per `CLAUDE.md`, and the finding is recorded rather than acted on: **the
calibrator choice should be n-dependent**, PAV for families with enough generators and logistic
below some threshold. That is a decision for the thesis, not for this notebook.

### 11.4 SENSITIVITY — 5-fold cross-fit, so every image carries an out-of-fold LR

The primary split leaves `autoregressive` and `graphics` with 9 val fakes each, at which point Cllr
is not reportable — a bootstrap CI at n=9 is wider than the range of values being compared.
`lr.cross_fit` stratifies folds by (label, generator), giving every one of the 1224 images an
out-of-fold LR while never calibrating on an image it scores, so those families are evaluated on 17
and fitted on about 13.

In [ ]:
Mi, OOF = lr.cross_fit(M, FAMILIES, l2s, k=5, seed=SEED)
print("fold sizes:", Mi.fold.value_counts().sort_index().tolist())

XFIT = report.crossfit_table(Mi, OOF, FAMILIES, FAMTAB)
print()
print(XFIT.round(4).to_string())

**Reading 11.4:** every family improves, and the calibration loss improves dramatically where the
split was starving the fit — `autoregressive` `Cllr_cal` falls from **0.344 to 0.057** and
`graphics` from **0.171 to 0.023**. Both families move from "worse than neutral" to genuinely
informative (`autoregressive` 1.234 → 0.761, `graphics` 1.092 → 0.834). `inpainting` drops below 1
for the first time, at 0.897.

The right reading is not "cross-fitting improves the system" — it is that **the primary split's
numbers for the three small families were dominated by fitting variance**, and the cross-fit
estimates are the ones to quote for them. The two large families barely move (gan 0.502 → 0.449,
diffusion 0.559 → 0.566), which is the check that the improvement is about sample size and not
about leakage sneaking in.

### 11.5 SENSITIVITY — covariate-matched band, `frac ∈ [0.10, 0.35)`

`crop_confound.ipynb` §3 restricted to this band and found every detector improved. Repeating it
here shows why it cannot serve as the primary analysis.

In [ ]:
band = M[(M.frac >= 0.10) & (M.frac < 0.35)]
print("composition inside the band:")
print(band.groupby("generator_family").size().to_string())
print("\nauthentic sources inside the band:")
print(band[band.label == 0].source.value_counts().to_string())
missing = [g for g in FAMILIES if (band.generator_family == g).sum() == 0]
print(f"\nunestimable under covariate matching (no images in band): {missing}")
print(f"RAISE images in band: {int((band.source == 'RAISE').sum())} of 204")

**Reading 11.5:** matching on the covariate annihilates the analysis. Three families —
`inpainting`, `autoregressive`, `graphics` — have **zero** images in the band, and `gan` is reduced
to 6. Only `diffusion` (48) and `other` (34) survive with usable counts.

And the authentic reference inside the band is COCO (203) plus part of LAION (83), with **zero of
the 204 RAISE images**. So the one analysis that controls the confound is also the one analysis that
structurally excludes the archival material the thesis exists to study. Reporting the band-restricted
Cllr as the headline would mean reporting a number validated on exactly the wrong population.

### 11.6 SENSITIVITY — residualizing on the covariate, reported as a trap

The tempting fix for §3 is to regress each detector score on `log10 frac` using authentic images and
work with the residual. It produces a large apparent improvement and it is **wrong**. The algebra:
fit `z ≈ a + b·log frac` on authentic rows, then the residual is `z − a − b·log frac`. The cell
below measures `b < 0` for four of five detectors, so `−b > 0` and the residual is

$$z_{\text{resid}} = z + |b|\cdot\log \text{frac} - a$$

— a **two-feature classifier that adds the confound back with the sign that helps**. Removing a
covariate effect estimated within one class does not de-confound when the covariate distributions
differ across classes; it re-confounds linearly. This is not adopted anywhere above.

In [ ]:
print("within-authentic regression of each sigma-score on the covariate:")
print(report.residualization_slopes(tr).round(4).to_string())
print("\n4 of 5 detectors have b < 0, so residualizing hands the classifier the confound back.")
print("Not adopted. The improvement it shows is an artifact of the differing covariate supports.")

## What this settles, and what it does not

In [ ]:
print(report.summary_table(MODS["per-family"], tr, FAMILIES, FAM_NGEN, FAM_DATE,
                           BLIND, FAMTAB, FLOOR, XFIT).round(3).to_string())

### The sensitivity statement

**A low LR is not evidence of authenticity.** This is not a caveat about this implementation; it is
what the quantity means. A properly calibrated LR satisfies `E[LR | H_d] = 1`, so an authentic
image's *expected* LR is 1, not 0 — and the measured rates of misleading evidence in §7 put
**10–22% of authentic images at `LR ≥ 1`** for every family. Symmetrically, `LR ≈ 1` on an image
means the system has nothing to say, and on this panel that happens to **64% of inpainting fakes**
and **67% of graphics fakes**. Low LR is consistent with an authentic image, with a degraded
synthetic image, and with a generator family the panel has never seen. The notebook cannot
distinguish those three, and neither can the thesis.

**Four specific limits, each measured above rather than asserted:**

1. **Under the source-count bound, the system cannot exonerate anything after 2020-12-17.**
   Measured in §9: the minimum `LR(F,T)` across all val images rises to exactly 1.0 on the date
   `autoregressive` (one generator, bound `[1, 1]`) joins `G≤T`, and stays there. A reader shown
   `LR = 1.0` on an authentic photograph after that date will read it as a finding about the
   photograph; it is a finding about the registry. The ELUB-bounded primary does not have this
   floor, so the choice of bound is not a detail — it decides whether the system can say anything
   exculpatory at all.

2. **Two of six families can support no likelihood ratio at all.** `autoregressive` is VQGAN alone
   and `graphics` is FaceSynthetics alone. Under the unit-of-analysis rule
   `temporal_correlation.ipynb` §5 established for this repo — the generator, not the image — the
   source-count bound is `[1, 1]` for both. This is a **registry-coverage** limit: no additional
   images per generator, and no better detector, changes it. Filling the post-2024-08 coverage gap
   `CLAUDE.md` names would not fix it either; what is needed is more generators *per family*.

3. **No LR reported here has been validated on the population that matters.** The authentic class
   is three disjoint populations in the crop-geometry covariate, and RAISE — the genuinely digitized
   archival material — overlaps **no** fake family (§3), so it is excluded from every
   covariate-controlled comparison (§11.5). `crop_confound.ipynb` §4 already measured RAISE
   false-flag rates of 11–61%. The archival domain gap that `CLAUDE.md` calls the thesis's central
   open question is therefore still open, and this notebook has not narrowed it.

4. **A metadata field beats the panel on half the families.** The crop-area fraction, which contains
   no image content whatsoever, reaches Cllr 0.055 on inpainting and 0.105 on graphics against the
   panel's 1.050 and 1.092 (§3, §6). Any LR quoted from this panel needs the geometry-only floor
   quoted beside it, or it overstates what the detectors contributed.

**On T\* specifically.** The crossing point is anti-conservative in the direction that matters:
median **3.3 years early at θ=2, 90% of fakes on the early side** (§10). Two independent mechanisms
drive it — the profile-LR multiplicity of taking a max over correlated families (§9,
`E[max | authentic]` = 4.21 against a properly calibrated 1.0) and the temporal leak of fitting a
family's model on generators released after T (§10, reduced but not removed in §11.2). Until both
are corrected, `T*(F)` from this pipeline should be read as a **lower bound on the date at which
fabrication became plausible**, never as an estimate of it. Pricing a timestamp against it would
systematically undervalue the anchor.

**What it does settle.** The architecture runs end to end on real data: the calibration layer
produces likelihood ratios with no prevalence assumption (§4), the bounds are derived from a
criterion rather than chosen (§4, §6), the aggregator is monotone by construction and two plausible
alternatives are shown to fail that test (§9), and the whole thing is measured with the
forensic-science battery `CLAUDE.md` specifies. The per-family fusion question is settled by proof
rather than preference (§5). And the two novel metrics the thesis proposes — temporal localization
error and the false-crossing rate on authentic material — are now implemented and give concrete,
uncomfortable numbers to argue with.